Import libraries

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import mean_absolute_error

Load and prepare the data

In [2]:
DATA_PATH = Path("../data/raw/SeoulBikeData.csv")

df = pd.read_csv(DATA_PATH, encoding="ISO-8859-1")

df = df.rename(
    columns={
        "Date": "date",
        "Rented Bike Count": "rented_bike_count",
        "Hour": "hour",
        "Temperature(°C)": "temperature",
        "Humidity(%)": "humidity",
        "Wind speed (m/s)": "wind_speed",
        "Visibility (10m)": "visibility",
        "Dew point temperature(°C)": "dew_point_temperature",
        "Solar Radiation (MJ/m2)": "solar_radiation",
        "Rainfall(mm)": "rainfall",
        "Snowfall (cm)": "snowfall",
        "Seasons": "season",
        "Holiday": "holiday",
        "Functioning Day": "functioning_day",
    }
)

df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")
df["timestamp"] = df["date"] + pd.to_timedelta(df["hour"], unit="h")
df = df.sort_values("timestamp").reset_index(drop=True)

df.head()

,date,rented_bike_count,hour,temperature,humidity,wind_speed,visibility,dew_point_temperature,solar_radiation,rainfall,snowfall,season,holiday,functioning_day,timestamp
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 00:00:00
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 01:00:00
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 02:00:00
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 03:00:00
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes,2017-12-01 04:00:00


Make a chronological split

In [3]:
train_end = int(len(df) * 0.70)
validation_end = int(len(df) * 0.85)

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))

Training rows: 6132
Validation rows: 1314
Test rows: 1314


Verify the periods

In [4]:
def show_period(name, data):
    print(
        f"{name}: "
        f"{data['timestamp'].min()} to "
        f"{data['timestamp'].max()}"
    )


show_period("Train", train_df)
show_period("Validation", validation_df)
show_period("Test", test_df)

Train: 2017-12-01 00:00:00 to 2018-08-13 11:00:00
Validation: 2018-08-13 12:00:00 to 2018-10-07 05:00:00
Test: 2018-10-07 06:00:00 to 2018-11-30 23:00:00


Build the mean baseline

In [5]:
target = "rented_bike_count"

training_mean = train_df[target].mean()

validation_predictions = [training_mean] * len(validation_df)

baseline_mae = mean_absolute_error(
    validation_df[target],
    validation_predictions,
)

print(f"Training mean: {training_mean:.2f}")
print(f"Baseline validation MAE: {baseline_mae:.2f}")

Training mean: 649.90
Baseline validation MAE: 596.76
